<a href="https://colab.research.google.com/github/rix031110/LLM_implementation/blob/MAria/RAG_reorganized.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Set-up

In [1]:
# Install required packages
!pip install -q -U sentence-transformers
!pip install -q transformers datasets faiss-cpu torch numpy pandas matplotlib seaborn scikit-learn pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.6/346.6 kB 29.0 MB/s eta 0:00:00


In [2]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Tuple
import time
import warnings
import os
warnings.filterwarnings('ignore')

from transformers import (
    AutoTokenizer,
    AutoModel,
    GPT2LMHeadModel,
    GPT2Tokenizer,
    T5Tokenizer,
    T5ForConditionalGeneration,
    set_seed
)
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
import faiss
from sklearn.metrics.pairwise import cosine_similarity
from pypdf import PdfReader

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## 2. Data import

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
def load_pdf_text(pdf_file_path: str) -> str:
    """
    Extract text from a PDF file, page by page.

    Each page is prefixed with a "--- Page N ---" marker so that the
    knowledge-base builder can later split the text back into pages.

    Args:
        pdf_file_path: Path to the PDF file

    Returns:
        Full extracted text as a single string
    """
    if not os.path.exists(pdf_file_path):
        raise FileNotFoundError(
            f"The file '{pdf_file_path}' was not found. "
            "Please ensure you have uploaded it correctly."
        )

    reader = PdfReader(pdf_file_path)
    num_pages = len(reader.pages)
    print(f"The PDF has {num_pages} page(s).")

    extracted_text = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text()
        if text:
            extracted_text.append(f"--- Page {i+1} ---\n{text}")
        else:
            extracted_text.append(f"--- Page {i+1} ---\n[No text found on this page]")

    return "\n".join(extracted_text)

In [5]:
# Load the English source document (default language used across most scenarios)
pdf_file_path = '/content/drive/MyDrive/LLM/ENG_REC.pdf'
full_text = load_pdf_text(pdf_file_path)

The PDF has 126 page(s).


## 3. Building the Knowledge Base

In [6]:
def build_knowledge_base(
    full_text: str,
    strategy: str = "paragraph",
    chunk_size: int = 600,
    overlap: int = 0,
    title_prefix: str = "FATF recommendations",
) -> List[Dict]:
    """
    Build a list of documents (the knowledge base) from the extracted PDF text.

    Three chunking strategies are supported through a single function so that
    every experimental scenario is just a different set of arguments:

      - strategy="page"      -> 1 page = 1 document
      - strategy="paragraph" -> word-based chunks built from paragraphs,
                                with an optional `overlap` (in words).
                                overlap=0  -> no overlap
                                overlap>0  -> sliding window with overlap

    Args:
        full_text: Text produced by load_pdf_text (with "--- Page N ---" markers)
        strategy: "page" or "paragraph"
        chunk_size: Maximum number of words per chunk (paragraph strategy only)
        overlap: Number of words shared between consecutive chunks (paragraph only)
        title_prefix: Prefix used for each document title

    Returns:
        List of documents, each a dict with keys: id, title, text
    """
    # Split full_text into individual pages
    page_contents = full_text.split('--- Page ')[1:]  # Skip the first empty split

    knowledge_base = []
    doc_id = 0

    if strategy == "page":
        # ---- 1 page = 1 document ----
        for page_content in page_contents:
            page_number_str, text_content = page_content.split(' ---\n', 1)
            page_number = int(page_number_str.strip())
            knowledge_base.append({
                'id': doc_id,
                'title': f'{title_prefix} (page {page_number})',
                'text': text_content.strip()
            })
            doc_id += 1

    elif strategy == "paragraph":
        # ---- word-based chunks built from paragraphs, with optional overlap ----
        for page_content in page_contents:
            page_number_str, text_content = page_content.split(' ---\n', 1)
            page_number = int(page_number_str.strip())
            text_content = text_content.strip()

            # Flatten the page into a single list of words coming from real paragraphs
            paragraphs = [p.strip() for p in text_content.split('\n') if p.strip()]
            words = []
            for para in paragraphs:
                words.extend(para.split())

            if not words:
                continue

            # Slide a window of `chunk_size` words, moving forward by step = chunk_size - overlap
            step = max(1, chunk_size - overlap)
            for start in range(0, len(words), step):
                sub = words[start:start + chunk_size]
                if not sub:
                    break
                knowledge_base.append({
                    'id': doc_id,
                    'title': f'{title_prefix} (page {page_number})',
                    'text': ' '.join(sub)
                })
                doc_id += 1
                # Stop once the window has reached the end of the page
                if start + chunk_size >= len(words):
                    break

    else:
        raise ValueError(f"Unknown strategy: '{strategy}'. Use 'page' or 'paragraph'.")

    print(f"Total documents created: {len(knowledge_base)} (strategy='{strategy}', chunk_size={chunk_size}, overlap={overlap})")
    return knowledge_base

## 4. RETRIEVAL - Building the Vector Database

### 4.1 Create Document Embeddings

In [7]:
# Load embedding model (Sentence-BERT) once; it is reused across all scenarios
print("Loading embedding model...")
# Option 1 Fast and efficient
#embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
# Option 2 fancier
embedding_model = SentenceTransformer(
    'Qwen/Qwen3-Embedding-0.6B',
    model_kwargs={"torch_dtype": "float16"},  # saves GPU
    tokenizer_kwargs={"padding_side": "left"},  # required by Qwen
)
embedding_dim = embedding_model.get_sentence_embedding_dimension()
print(f"Embedding model loaded. Dimension: {embedding_dim}")



Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Embedding model loaded. Dimension: 1024


In [8]:
def embed_documents(knowledge_base: List[Dict], embedding_model) -> Tuple[pd.DataFrame, np.ndarray]:
    """
    Turn a knowledge base into a DataFrame and a matrix of document embeddings.

    Args:
        knowledge_base: List of documents (id, title, text)
        embedding_model: SentenceTransformer used to encode the documents

    Returns:
        (df_kb, doc_embeddings) where df_kb is the documents as a DataFrame
        and doc_embeddings is a float32 array of normalized-ready vectors.
    """
    df_kb = pd.DataFrame(knowledge_base)

    # Combine title and text for richer embeddings
    df_kb['combined_text'] = df_kb['title'] + ': ' + df_kb['text']
    documents = df_kb['combined_text'].tolist()

    start_time = time.time()
    doc_embeddings = embedding_model.encode(documents, show_progress_bar=False)
    doc_embeddings = np.array(doc_embeddings).astype('float32')
    embedding_time = time.time() - start_time

    print(f"Embeddings generated in {embedding_time:.2f} seconds | shape: {doc_embeddings.shape}")
    return df_kb, doc_embeddings

### 4.2 Build FAISS Index

In [9]:
def build_faiss_index(doc_embeddings: np.ndarray) -> faiss.Index:
    """
    Build a FAISS index from document embeddings.

    Vectors are L2-normalized so that the L2 index behaves like cosine similarity.

    Args:
        doc_embeddings: float32 array of document embeddings

    Returns:
        A FAISS index containing the (normalized) document vectors.
    """
    embedding_dim = doc_embeddings.shape[1]
    index = faiss.IndexFlatL2(embedding_dim)  # L2 distance (Euclidean)

    # Normalize vectors for cosine similarity
    faiss.normalize_L2(doc_embeddings)
    index.add(doc_embeddings)
    return index

### 4.3 Implement Retrieval Function

In [10]:
def retrieve_documents(query: str, index, df_kb: pd.DataFrame,
                       embedding_model, k: int = 3) -> List[Dict]:
    """
    Retrieve the top-k most relevant documents for a query.

    The index, df_kb and embedding_model are passed in explicitly (instead of
    relying on globals) so that each scenario is self-contained and reproducible.

    Args:
        query: The search query
        index: FAISS index to search in
        df_kb: DataFrame of documents matching the index order
        embedding_model: SentenceTransformer used to encode the query
        k: Number of documents to retrieve

    Returns:
        List of retrieved documents, each including a 'retrieval_score'.
    """
    # Encode query

    # For option 1
    #query_embedding = embedding_model.encode([query])
    # For option 2
    query_embedding = embedding_model.encode([query], prompt_name="query")
    query_embedding = np.array(query_embedding).astype('float32')
    faiss.normalize_L2(query_embedding)

    # Search in FAISS index
    distances, indices = index.search(query_embedding, k)

    # Convert L2 distance on normalized vectors to a cosine-similarity-like score
    similarities = 1 - distances[0]

    # Retrieve documents
    results = []
    for idx, score in zip(indices[0], similarities):
        doc = df_kb.iloc[idx].to_dict()
        doc['retrieval_score'] = float(score)
        results.append(doc)

    return results

### 4.4 Test Retrieval

In [16]:
# Quick smoke test of the retrieval stage on a paragraph-based knowledge base.
# (Built here only to verify retrieval works; the experiment scenarios rebuild
#  their own knowledge bases later.)
_kb_demo = build_knowledge_base(full_text, strategy="paragraph", chunk_size=600, overlap=0)
_df_demo, _emb_demo = embed_documents(_kb_demo, embedding_model)
_index_demo = build_faiss_index(_emb_demo)

_demo_query = "What is the Financial Action Task Force?"
print(f"\nQuery: '{_demo_query}'\n")
for i, doc in enumerate(retrieve_documents(_demo_query, _index_demo, _df_demo, embedding_model, k=3), 1):
    print(f"{i}. {doc['title']} (score: {doc['retrieval_score']:.3f})")
    print(f"   {doc['text'][:100]}...\n")

Total documents created: 126 (strategy='paragraph', chunk_size=600, overlap=0)
Embeddings generated in 6.90 seconds | shape: (126, 1024)

Query: 'What is the Financial Action Task Force?'

1. FATF recommendations (page 2) (score: 0.477)
   FINANCIAL ACTION TASK FORCE The Financial Action Task Force (FATF) is an independent inter -governme...

2. FATF recommendations (page 9) (score: 0.351)
   THE FATF RECOMMENDATIONS INTERNATIONAL STANDARDS ON COMBATING MONEY LAUNDERING AND THE FINANCING OF ...

3. FATF recommendations (page 125) (score: 0.300)
   THE FATF RECOMMENDATIONS INTERNATIONAL STANDARDS ON COMBATING MONEY LAUNDERING AND THE FINANCING OF ...



## 5. AUGMENTATION - Building the Prompt

In [17]:
def create_augmented_prompt(query: str, retrieved_docs: List[Dict],
                            max_context_length: int = 900) -> str:
    """
    Create an augmented prompt by combining the query with retrieved context.

    Args:
        query: User's question
        retrieved_docs: List of retrieved documents
        max_context_length: Maximum number of characters used for the context

    Returns:
        Augmented prompt string
    """
    # Build context from retrieved documents, respecting the character budget
    context_parts = []
    current_length = 0

    for doc in retrieved_docs:
        doc_text = f"[{doc['title']}] {doc['text']}"
        if current_length + len(doc_text) <= max_context_length:
            context_parts.append(doc_text)
            current_length += len(doc_text)
        else:
            break

    context = "\n\n".join(context_parts)

    # Instruction-style prompt (works well for Flan-T5 and is harmless for GPT-2)
    prompt = f"""Answer the question using only the context below.

Context: {context}
Question: {query}
Answer:"""

    return prompt

In [19]:
# Illustrative example of an augmented prompt (shown once, for reference only).
_example_query = "What is the Financial Action Task Force? "
_example_retrieved = retrieve_documents(_example_query, _index_demo, _df_demo, embedding_model, k=2)
print(create_augmented_prompt(_example_query, _example_retrieved))

Answer the question using only the context below.

Context: [FATF recommendations (page 2)] FINANCIAL ACTION TASK FORCE The Financial Action Task Force (FATF) is an independent inter -governmental body that develops and promotes policies to protect the global financial system against money laundering, terrorist financing and the financing of proliferation of weapons of mass destruction . The FATF Recommendations are recognised as the global anti-money laundering (AML) and counter-terrorist financing (CFT) standard. For more information about the FATF, please visit the website: www.fatf-gafi.org © 2012 FATF/OECD. All rights reserved. No reproduction or translation of this publication may be made without prior written permission. Applications for such permission, for all or part of this publication, should be made to the FATF Secretariat, 2 rue André Pascal 75775 Paris Cedex 16, France (fax: +33 1 44 30 61 37 or e-mail: contact@fatf-gafi.org).
Question: What is the Financial Action Task 

## 6. GENERATION - Producing the Answer

In [20]:
def load_generator(model_type: str):
    """
    Load a generation model and its tokenizer.

    Args:
        model_type: "gpt2" (causal LM) or "flan-t5" (seq2seq LM)

    Returns:
        (tokenizer, model) tuple, with the model moved to `device`.
    """
    print(f"Loading generation model: {model_type} ...")
    if model_type == "gpt2":
        tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
        model = GPT2LMHeadModel.from_pretrained('gpt2').to(device)
        tokenizer.pad_token = tokenizer.eos_token
    elif model_type == "flan-t5":
        tokenizer = T5Tokenizer.from_pretrained('google/flan-t5-base')
        model = T5ForConditionalGeneration.from_pretrained('google/flan-t5-base').to(device)
    else:
        raise ValueError(f"Unknown model_type: '{model_type}'. Use 'gpt2' or 'flan-t5'.")
    print("Generation model loaded.")
    return tokenizer, model

In [26]:
def generate_answer(prompt: str, tokenizer, model, model_type: str,
                    max_new_tokens: int = 200, seed: int = 42) -> str:
    """
    Generate an answer from an augmented prompt.
    A fixed seed makes each call reproducible regardless of call order.

    Handles both model families:
      - GPT-2 (causal): output contains the prompt, so we strip it and we must
        keep prompt + generation within the 1024-token context window.
      - Flan-T5 (seq2seq): output IS the answer; input is truncated to 512 tokens.

    Args:
        prompt: The augmented prompt with context
        tokenizer, model: Returned by load_generator
        model_type: "gpt2" or "flan-t5"
        max_new_tokens: Maximum number of new tokens to generate

    Returns:
        Generated answer string
    """
    set_seed(seed)  # reset RNG state before sampling for reproducibility

    if model_type == "gpt2":
        # Leave room for the generated tokens inside GPT-2's 1024-token window
        max_input_tokens = 1024 - max_new_tokens
        input_ids = tokenizer.encode(
            prompt, return_tensors='pt',
            truncation=True, max_length=max_input_tokens
        ).to(device)

        output = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            num_return_sequences=1,
            pad_token_id=tokenizer.eos_token_id
        )

        full_text = tokenizer.decode(output[0], skip_special_tokens=True)
        # The answer is whatever comes after the prompt
        answer = full_text[len(prompt):].strip()
        return answer

    elif model_type == "flan-t5":
        input_ids = tokenizer.encode(
            prompt, return_tensors='pt',
            truncation=True, max_length=512
        ).to(device)

        output = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            num_return_sequences=1
        )

        # In seq2seq the output is ONLY the answer (it does not include the prompt)
        return tokenizer.decode(output[0], skip_special_tokens=True).strip()

    else:
        raise ValueError(f"Unknown model_type: '{model_type}'. Use 'gpt2' or 'flan-t5'.")

### RAG pipeline and scenario runner

In [21]:
def rag_pipeline(query: str, index, df_kb, embedding_model,
                 tokenizer, model, model_type: str, k: int = 3) -> Dict:
    """
    Complete RAG pipeline for a single query: Retrieve -> Augment -> Generate.

    All resources (index, df_kb, models) are passed in so the pipeline is
    self-contained and can be reused across scenarios.

    Returns:
        Dictionary with the query, generated answer, retrieved docs and timings.
    """
    start_time = time.time()

    # Step 1: Retrieve
    retrieval_start = time.time()
    retrieved_docs = retrieve_documents(query, index, df_kb, embedding_model, k=k)
    retrieval_time = time.time() - retrieval_start

    # Step 2: Augment
    augment_start = time.time()
    augmented_prompt = create_augmented_prompt(query, retrieved_docs)
    augment_time = time.time() - augment_start

    # Step 3: Generate
    gen_start = time.time()
    answer = generate_answer(augmented_prompt, tokenizer, model, model_type)
    gen_time = time.time() - gen_start

    total_time = time.time() - start_time

    return {
        'query': query,
        'answer': answer,
        'retrieved_docs': retrieved_docs,
        'augmented_prompt': augmented_prompt,
        'timings': {
            'retrieval': retrieval_time,
            'augmentation': augment_time,
            'generation': gen_time,
            'total': total_time
        }
    }

In [22]:
def is_match(expected: str, generated: str) -> bool:
    """
    Simple factual-match check for short answers: True if the expected answer
    (case-insensitive) appears inside the generated answer.
    """
    return expected.lower().strip() in generated.lower().strip()


def run_scenario(full_text: str, strategy: str, model_type: str,
                 questions: List[str], expected_answers: List[str],
                 embedding_model, tokenizer, model,
                 chunk_size: int = 600, overlap: int = 0, k: int = 2,
                 title_prefix: str = "FATF recommendations") -> pd.DataFrame:
    """
    Run the full RAG flow for one scenario over all questions and return a
    per-question results DataFrame.

    A "scenario" = a chunking strategy + a generation model. The embedding model
    and the generation model are passed in (loaded once, reused) to save time.

    Returns:
        DataFrame with columns: question, expected, generated, match, gen_time_s
    """
    # Build a fresh knowledge base + index for this chunking strategy
    kb = build_knowledge_base(full_text, strategy=strategy,
                              chunk_size=chunk_size, overlap=overlap,
                              title_prefix=title_prefix)
    df_kb, doc_embeddings = embed_documents(kb, embedding_model)
    index = build_faiss_index(doc_embeddings)

    rows = []
    for question, expected in zip(questions, expected_answers):
        result = rag_pipeline(question, index, df_kb, embedding_model,
                              tokenizer, model, model_type, k=k)
        rows.append({
            'question': question,
            'expected': expected,
            'generated': result['answer'],
            'match': is_match(expected, result['answer']),
            'gen_time_s': round(result['timings']['generation'], 3),
        })

    return pd.DataFrame(rows)

### Evaluation questions and expected answers

In [33]:
# 8 evaluation questions and their expected (ground-truth) answers
questions = [
    "What is the Financial Action Task Force?",
    "What is FATF?",
    "What does AML/CFT mean?",
    "When were the recommendations first drafted?",
    "On which year were the recommendations last updated?",
    "Why are Non-profit organisations included on the recommendations?",
    "Why are the FATF recommendations being revised?",
    "Why should countries implement a risk-based approach?",
]

expected_answers = [
    "an inter-governmental body",
    "the Financial Action Task Force",
    "anti-money laundering and countering the financing of terrorism",
    "1996",
    "2003",
    "they are particularly vulnerable to misuse",
    "to address new and emerging threats",
    "to target their resources more effectively",
]

### Load both generation models (once)

In [34]:
# Load both generators once so scenarios only switch between them, no reloads
gpt2_tokenizer, gpt2_model = load_generator("gpt2")
t5_tokenizer, t5_model = load_generator("flan-t5")

Loading generation model: gpt2 ...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Generation model loaded.
Loading generation model: flan-t5 ...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Generation model loaded.


### Scenarios

| # | Chunking strategy | Generation model |
|---|-------------------|------------------|
| 1 | page              | GPT-2            |
| 2 | paragraph (300w, no overlap)   | GPT-2 |
| 3 | paragraph (300w, overlap=50)   | GPT-2 |
| 4 | page              | Flan-T5          |
| 5 | paragraph (300w, no overlap)   | Flan-T5 |
| 6 | paragraph (300w, overlap=50)   | Flan-T5 |


In [35]:
# Scenario 1: page = 1 document, GPT-2
df_s1 = run_scenario(full_text, strategy="page", model_type="gpt2",
                     questions=questions, expected_answers=expected_answers,
                     embedding_model=embedding_model,
                     tokenizer=gpt2_tokenizer, model=gpt2_model)
df_s1

Total documents created: 126 (strategy='page', chunk_size=600, overlap=0)
Embeddings generated in 7.56 seconds | shape: (126, 1024)


,question,expected,generated,match,gen_time_s
0,What is the Financial Action Task Force?,an inter-governmental body,"This is the Financial Action Task Force, and i...",False,1.950
1,What is FATF?,the Financial Action Task Force,It is a simple application that allows you to ...,False,1.960
2,What does AML/CFT mean?,anti-money laundering and countering the finan...,AML/CFT is the term used to describe the relat...,False,1.918
3,When were the recommendations first drafted?,1996,"In the spring of 2008, the FATF had published ...",False,2.178
4,On which year were the recommendations last up...,2003,2013\n\nQuestion: On which year are the recomm...,False,2.638
5,Why are Non-profit organisations included on t...,they are particularly vulnerable to misuse,"This question is a very important one, and it ...",False,1.912
6,Why are the FATF recommendations being revised?,to address new and emerging threats,This question is being asked as part of the FA...,False,1.885
7,Why should countries implement a risk-based ap...,to target their resources more effectively,This question is a classic example of a questi...,False,1.925


No answer is correct

In [36]:
# Scenario 2: paragraph (300w, no overlap), GPT-2
df_s2 = run_scenario(full_text, strategy="paragraph", model_type="gpt2",
                     questions=questions, expected_answers=expected_answers,
                     embedding_model=embedding_model,
                     tokenizer=gpt2_tokenizer, model=gpt2_model,
                     chunk_size=600, overlap=0)
df_s2

Total documents created: 126 (strategy='paragraph', chunk_size=600, overlap=0)
Embeddings generated in 6.94 seconds | shape: (126, 1024)


,question,expected,generated,match,gen_time_s
0,What is the Financial Action Task Force?,an inter-governmental body,The FATF Task Force is an independent inter -g...,False,2.775
1,What is FATF?,the Financial Action Task Force,FATF is the Financial Action Task Force (FATF)...,True,2.335
2,What does AML/CFT mean?,anti-money laundering and countering the finan...,AML/CFT is the term used to describe the relat...,False,2.006
3,When were the recommendations first drafted?,1996,"In the spring of 2008, the FATF had published ...",False,1.927
4,On which year were the recommendations last up...,2003,2013\n\nQuestion: On which year are the recomm...,False,1.913
5,Why are Non-profit organisations included on t...,they are particularly vulnerable to misuse,"This question is a very important one, and it ...",False,1.940
6,Why are the FATF recommendations being revised?,to address new and emerging threats,This question is being asked as part of the FA...,False,2.462
7,Why should countries implement a risk-based ap...,to target their resources more effectively,This question is a classic example of a questi...,False,2.600


Answer to Q0 is also correct.

In [37]:
# Scenario 3: paragraph (300w, overlap=50), GPT-2
df_s3 = run_scenario(full_text, strategy="paragraph", model_type="gpt2",
                     questions=questions, expected_answers=expected_answers,
                     embedding_model=embedding_model,
                     tokenizer=gpt2_tokenizer, model=gpt2_model,
                     chunk_size=600, overlap=50)
df_s3

Total documents created: 126 (strategy='paragraph', chunk_size=600, overlap=50)
Embeddings generated in 7.16 seconds | shape: (126, 1024)


,question,expected,generated,match,gen_time_s
0,What is the Financial Action Task Force?,an inter-governmental body,The FATF Task Force is an independent inter -g...,False,1.907
1,What is FATF?,the Financial Action Task Force,FATF is the Financial Action Task Force (FATF)...,True,1.904
2,What does AML/CFT mean?,anti-money laundering and countering the finan...,AML/CFT is the term used to describe the relat...,False,2.582
3,When were the recommendations first drafted?,1996,"In the spring of 2008, the FATF had published ...",False,2.282
4,On which year were the recommendations last up...,2003,2013\n\nQuestion: On which year are the recomm...,False,1.901
5,Why are Non-profit organisations included on t...,they are particularly vulnerable to misuse,"This question is a very important one, and it ...",False,1.888
6,Why are the FATF recommendations being revised?,to address new and emerging threats,This question is being asked as part of the FA...,False,1.913
7,Why should countries implement a risk-based ap...,to target their resources more effectively,This question is a classic example of a questi...,False,1.884


Answer to Q0 is also true

In [38]:
# Scenario 4: page = 1 document, Flan-T5
df_s4 = run_scenario(full_text, strategy="page", model_type="flan-t5",
                     questions=questions, expected_answers=expected_answers,
                     embedding_model=embedding_model,
                     tokenizer=t5_tokenizer, model=t5_model)
df_s4

Total documents created: 126 (strategy='page', chunk_size=600, overlap=0)
Embeddings generated in 7.92 seconds | shape: (126, 1024)


,question,expected,generated,match,gen_time_s
0,What is the Financial Action Task Force?,an inter-governmental body,organization,False,0.077
1,What is FATF?,the Financial Action Task Force,A federal agency that regulates football and f...,False,0.381
2,What does AML/CFT mean?,anti-money laundering and countering the finan...,AML/CFT,False,0.203
3,When were the recommendations first drafted?,1996,February 2012,False,0.121
4,On which year were the recommendations last up...,2003,2017,False,0.066
5,Why are Non-profit organisations included on t...,they are particularly vulnerable to misuse,to improve the health of their population,False,0.163
6,Why are the FATF recommendations being revised?,to address new and emerging threats,to improve the health of patients,False,0.144
7,Why should countries implement a risk-based ap...,to target their resources more effectively,To avoid economic collapse,False,0.122


No correct answers

In [39]:
# Scenario 5: paragraph (300w, no overlap), Flan-T5
df_s5 = run_scenario(full_text, strategy="paragraph", model_type="flan-t5",
                     questions=questions, expected_answers=expected_answers,
                     embedding_model=embedding_model,
                     tokenizer=t5_tokenizer, model=t5_model,
                     chunk_size=600, overlap=0)
df_s5

Total documents created: 126 (strategy='paragraph', chunk_size=600, overlap=0)
Embeddings generated in 7.43 seconds | shape: (126, 1024)


,question,expected,generated,match,gen_time_s
0,What is the Financial Action Task Force?,an inter-governmental body,independent inter -governmental body,False,0.150
1,What is FATF?,the Financial Action Task Force,Financial Action Task Force,False,0.110
2,What does AML/CFT mean?,anti-money laundering and countering the finan...,AML/CFT,False,0.133
3,When were the recommendations first drafted?,1996,2012,False,0.057
4,On which year were the recommendations last up...,2003,2017,False,0.056
5,Why are Non-profit organisations included on t...,they are particularly vulnerable to misuse,to improve the health of their population,False,0.181
6,Why are the FATF recommendations being revised?,to address new and emerging threats,to improve the health of patients,False,0.148
7,Why should countries implement a risk-based ap...,to target their resources more effectively,To avoid economic collapse,False,0.109


Answers to Q0 and Q1 are correct.

In [40]:
# Scenario 6: paragraph (300w, overlap=50), Flan-T5
df_s6 = run_scenario(full_text, strategy="paragraph", model_type="flan-t5",
                     questions=questions, expected_answers=expected_answers,
                     embedding_model=embedding_model,
                     tokenizer=t5_tokenizer, model=t5_model,
                     chunk_size=600, overlap=50)
df_s6

Total documents created: 126 (strategy='paragraph', chunk_size=600, overlap=50)
Embeddings generated in 7.43 seconds | shape: (126, 1024)


,question,expected,generated,match,gen_time_s
0,What is the Financial Action Task Force?,an inter-governmental body,independent inter -governmental body,False,0.158
1,What is FATF?,the Financial Action Task Force,Financial Action Task Force,False,0.115
2,What does AML/CFT mean?,anti-money laundering and countering the finan...,AML/CFT,False,0.127
3,When were the recommendations first drafted?,1996,2012,False,0.055
4,On which year were the recommendations last up...,2003,2017,False,0.055
5,Why are Non-profit organisations included on t...,they are particularly vulnerable to misuse,to improve the health of their population,False,0.164
6,Why are the FATF recommendations being revised?,to address new and emerging threats,to improve the health of patients,False,0.151
7,Why should countries implement a risk-based ap...,to target their resources more effectively,To avoid economic collapse,False,0.129


Answer to Q0 and Q1 are correct.

### Summary table: matches per scenario

In [41]:
#manually change false/true values according to validation
df_s2.loc[0, 'match'] = True
df_s3.loc[0, 'match'] = True
df_s5.loc[0, 'match'] = True
df_s5.loc[1, 'match'] = True
df_s6.loc[0, 'match'] = True
df_s6.loc[1, 'match'] = True

In [42]:
# Aggregate how many of the 8 questions each scenario answered correctly
scenario_labels = {
    "Scenario 1": ("page",               "GPT-2",   df_s1),
    "Scenario 2": ("paragraph no-overlap", "GPT-2", df_s2),
    "Scenario 3": ("paragraph overlap=50", "GPT-2", df_s3),
    "Scenario 4": ("page",               "Flan-T5", df_s4),
    "Scenario 5": ("paragraph no overlap", "Flan-T5", df_s5),
    "Scenario 6": ("paragraph overlap=50", "Flan-T5", df_s6),
}

summary = pd.DataFrame([
    {
        'scenario': name,
        'chunking': chunking,
        'model': model_name,
        'matches': int(df['match'].sum()),
        'accuracy (%)': round((df['match'].mean())*100, 2),
    }
    for name, (chunking, model_name, df) in scenario_labels.items()
])
summary

,scenario,chunking,model,matches,accuracy (%)
0,Scenario 1,page,GPT-2,0,0.0
1,Scenario 2,paragraph no-overlap,GPT-2,2,25.0
2,Scenario 3,paragraph overlap=50,GPT-2,2,25.0
3,Scenario 4,page,Flan-T5,0,0.0
4,Scenario 5,paragraph no overlap,Flan-T5,2,25.0
5,Scenario 6,paragraph overlap=50,Flan-T5,2,25.0
